In [ ]:
%matplotlib inline
from data_loader import *
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from datetime import date, timedelta


# PARTE 4 — Análise Bivariada

> Esta seção aprofunda a análise univariada anterior, cruzando variáveis para revelar tensões estratégicas que não são visíveis em distribuições isoladas. O objetivo é transformar dados em decisões concretas para o problema de abastecimento Malzbier Brahma na região NENO (Nordeste) em fevereiro de 2026.

## Por que análise bivariada?

A análise univariada mostrou que existe um **gap de 4.500 HL/mês** de Malzbier Brahma na região NENO. Mas ela não responde: *de onde vem o volume? a que custo? em quanto tempo? com que risco?*

Para isso, precisamos cruzar variáveis:

| # | Cruzamento | Tensão Estratégica | Pergunta-Guia | Decisão Revelada |
|---|---|---|---|---|
| 31 | Produção × Demanda | Capacidade ociosa vs. necessidade urgente | Há espaço produtivo não utilizado que pode cobrir o gap? | Reprojetar Malzbier em PE541 W1 |
| 32 | Custo × Geografia | Modal mais barato vs. CDR mais urgente | Qual rota oferece melhor margem residual por sub-região? | Priorizar Cabotagem Fonte Mata (João Pessoa) |
| 33 | Capacidade × Tempo | Lead time do modal vs. janela de ruptura | Qual modal chega antes do DOI atingir zero? | Descartar Cabotagem para fevereiro |
| 34 | Risco × Retorno | BIAS histórico vs. rentabilidade dos cenários | Qual cenário permanece lucrativo mesmo com over-forecast? | Cenário C é o único robusto a BIAS>9% |
| 35 | Tático × Estratégico | Decisão emergencial vs. posicionamento futuro | A decisão de fevereiro é boa também para março-junho? | Investimento de R$458k gera ROI ~730% até junho |


---
## 31. Cruzamento 1 — Produção × Demanda

> **Pergunta:** Existe capacidade produtiva ociosa que pode cobrir o gap de Malzbier sem investimento em nova capacidade?

### Contexto

O gap semanal de Malzbier é de **1.125 HL/semana** (4.500 HL ÷ 4 semanas). A pergunta é: as plantas já instaladas — AQ541 (Aquiraz/CE, cap. 12.600 HL/sem) e PE541 (Nassau/PE, cap. 27.000 HL/sem) — têm espaço para absorver esse volume sem comprometer outros SKUs?

A resposta está na **PE541 W1**: a planta opera a apenas 73,3% de sua capacidade (19.800 HL de 27.000 HL possíveis), com **7.200 HL completamente ociosos** — e **Malzbier está zerado** no planejamento da W1.

| Planta | Cap/sem (HL) | Malzbier atual W1 (HL) | Ociosidade W1 (HL) | Potencial de reprojeção |
|---|---|---|---|---|
| AQ541 (CE) | 12.600 | Ver df_pcp | Variável | Limitado — Patagonia domina |
| PE541 (PE) | 27.000 | **0** | **7.200** | **Cobre 640% do gap semanal** |

### Insight Crítico

7.200 HL de ociosidade em PE541 W1 cobre **640% do gap semanal** (1.125 HL) — ou seja, um único ajuste de planejamento resolve o mês inteiro sem precisar de nenhuma transferência emergencial.

**Restrição:** Goose Island **NÃO pode** ser movido em PE541 por restrição de elaboração de líquido. Portanto, a ociosidade disponível é real e utilizável para Malzbier.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 31 — Produção × Demanda
# ═══════════════════════════════════════════════════════════════════════

# ── Preparação de dados ──────────────────────────────────────────────────
prod_aq = df_pcp[df_pcp['Planta'] == 'AQ541 (CE)'].copy()
prod_pe = df_pcp[df_pcp['Planta'] == 'PE541 (PE)'].copy()

semana_nums = [1, 2, 3, 4]

# Pivot: linhas = SKU, colunas = Semana_Num
def get_pivot(df_planta):
    piv = df_planta.pivot_table(index='SKU', columns='Semana_Num', values='Volume_HL', aggfunc='sum').fillna(0)
    piv = piv.reindex(columns=semana_nums, fill_value=0)
    return piv

piv_aq = get_pivot(prod_aq)
piv_pe = get_pivot(prod_pe)

# Total por semana
total_aq = piv_aq.sum(axis=0)
total_pe = piv_pe.sum(axis=0)

# Ociosidade PE541
ocio_pe = pd.Series([cap_pe - total_pe[w] for w in semana_nums], index=semana_nums)

# Demanda Malzbier NENO (nova demanda)
malz_nova = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['Demanda'].sum().reindex(semana_nums, fill_value=0)
malz_div  = div_neno[div_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['Demanda'].sum().reindex(semana_nums, fill_value=0)

# Produção Malzbier nas plantas
malz_prod_pe = piv_pe.loc['Malzbier'] if 'Malzbier' in piv_pe.index else pd.Series(0, index=semana_nums)
malz_prod_aq = piv_aq.loc['Malzbier'] if 'Malzbier' in piv_aq.index else pd.Series(0, index=semana_nums)
malz_prod_total = malz_prod_pe.add(malz_prod_aq, fill_value=0)

# Gap semanal
gap_semana = malz_nova - malz_prod_total

# Simulação DOI
# DOI = EF_Semana / (Demanda_semana / 7)
# Sem ação: produção conforme PCP
# Com ação: adicionar 4500 HL na W1 (max ociosidade disponível para cobrir gap)
malz_ei = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['EI_Semana'].sum().reindex(semana_nums, fill_value=0)
malz_ef = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['EF_Semana'].sum().reindex(semana_nums, fill_value=0)
malz_suf_f = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['Suf_Final_dias'].sum().reindex(semana_nums, fill_value=0)

doi_sem_acao = malz_suf_f.copy()
# Com ação: adicionar 4500 HL na W1 e distribuir uniformemente
adicional_w1 = 4500
doi_com_acao = malz_suf_f.copy()
# Simulação simples: acrescentar volume ao estoque final
dem_media_sem = malz_nova / 7  # HL/dia
for w in semana_nums:
    extra = adicional_w1 if w == 1 else 0
    ef_adj = malz_ef[w] + extra
    doi_com_acao[w] = ef_adj / dem_media_sem[w] if dem_media_sem[w] > 0 else 0

print("=" * 70)
print("CRUZAMENTO 31: PRODUÇÃO × DEMANDA — TABELA RESUMO")
print("=" * 70)
print(f"{'Semana':<15} {'Prod Malz (HL)':>16} {'Dem Nova (HL)':>14} {'Gap (HL)':>10} {'DOI s/ação':>11} {'DOI c/ação':>11}")
print("-" * 70)
for w in semana_nums:
    print(f"{'W' + str(w) + ' ' + semanas[w-1][2:]:<15} {malz_prod_total[w]:>16,.0f} {malz_nova[w]:>14,.0f} {gap_semana[w]:>10,.0f} {doi_sem_acao[w]:>11.1f} {doi_com_acao[w]:>11.1f}")
print("=" * 70)
print(f"\nOciosidade PE541 W1: {ocio_pe[1]:,.0f} HL (cobre {ocio_pe[1]/1125:.1f}x o gap semanal)")
print(f"Gap total mensal: {gap_semana.sum():,.0f} HL")

# ── Figura ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Seção 31 — Produção × Demanda: Capacidade Ociosa PE541 vs. Gap Malzbier', 
             fontsize=14, fontweight='bold', color=CORES['azul_escuro'], y=1.01)

x = np.arange(4)
w_labels = [s[:2] for s in semanas]
bar_w = 0.35

# ── Painel 1: Produção semanal AQ541 vs PE541 por SKU ─────────────────────
ax1 = axes[0, 0]
skus_aq = piv_aq.index.tolist()
skus_pe = piv_pe.index.tolist()
cores_skus = [CORES['azul_escuro'], CORES['azul_medio'], CORES['azul_claro'], 
              CORES['ambar'], CORES['verde'], CORES['vermelho'], CORES['cinza_medio']]

bottom_aq = np.zeros(4)
for i, sku in enumerate(skus_aq):
    vals = [piv_aq.loc[sku, w] for w in semana_nums]
    ax1.bar(x - bar_w/2, vals, bar_w, bottom=bottom_aq, 
            color=cores_skus[i % len(cores_skus)], label=f'AQ-{sku}', alpha=0.9)
    bottom_aq += np.array(vals)

bottom_pe = np.zeros(4)
for i, sku in enumerate(skus_pe):
    vals = [piv_pe.loc[sku, w] for w in semana_nums]
    ax1.bar(x + bar_w/2, vals, bar_w, bottom=bottom_pe, 
            color=cores_skus[i % len(cores_skus)], alpha=0.6, hatch='//')
    bottom_pe += np.array(vals)

ax1.axhline(cap_aq, color=CORES['azul_medio'], linestyle='--', linewidth=1.5, label=f'Cap AQ541={cap_aq:,}')
ax1.axhline(cap_pe, color=CORES['ambar'], linestyle='--', linewidth=1.5, label=f'Cap PE541={cap_pe:,}')
ax1.set_title('Produção por Planta e SKU (barras sólidas=AQ541, hachuras=PE541)', fontsize=10)
ax1.set_xticks(x)
ax1.set_xticklabels(w_labels)
ax1.set_ylabel('Volume (HL)')
ax1.legend(fontsize=7, ncol=2)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Painel 2: PE541 — Capacidade vs Produção com ociosidade ───────────────
ax2 = axes[0, 1]
total_pe_list = [total_pe[w] for w in semana_nums]
ocio_pe_list = [ocio_pe[w] for w in semana_nums]
bars_prod = ax2.bar(x, total_pe_list, color=CORES['azul_medio'], label='Produção Total PE541', alpha=0.85)
bars_ocio = ax2.bar(x, ocio_pe_list, bottom=total_pe_list, color=CORES['ambar'], alpha=0.5, label='Ociosidade PE541')
ax2.axhline(cap_pe, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'Capacidade máxima ({cap_pe:,} HL)')
ax2.axhline(1125, color=CORES['verde'], linestyle=':', linewidth=1.5, label='Gap semanal (1.125 HL)')

# Anotação W1
ax2.annotate(f'7.200 HL\nociosos\n(W1)', xy=(0, total_pe_list[0] + ocio_pe_list[0]/2),
             xytext=(0.5, total_pe_list[0] + ocio_pe_list[0]/2 + 1000),
             fontsize=9, color=CORES['vermelho'], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=CORES['vermelho']))
ax2.set_title('PE541 — Capacidade × Utilização (W1 tem 7.200 HL ociosos)', fontsize=10)
ax2.set_xticks(x)
ax2.set_xticklabels(w_labels)
ax2.set_ylabel('Volume (HL)')
ax2.legend(fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Painel 3: Gap Malzbier por semana ─────────────────────────────────────
ax3 = axes[1, 0]
ax3.fill_between(semana_nums, malz_div.values, malz_prod_total.values,
                 where=(malz_div.values > malz_prod_total.values),
                 alpha=0.3, color=CORES['vermelho'], label='Gap (divulgado)')
ax3.fill_between(semana_nums, malz_nova.values, malz_prod_total.values,
                 where=(malz_nova.values > malz_prod_total.values),
                 alpha=0.4, color=CORES['ambar'], label='Gap adicional (nova dem.)')
ax3.plot(semana_nums, malz_div.values, 'o--', color=CORES['azul_medio'], linewidth=2, label='Demanda Divulgada')
ax3.plot(semana_nums, malz_nova.values, 's-', color=CORES['ambar'], linewidth=2.5, label='Nova Demanda (+30%)')
ax3.plot(semana_nums, malz_prod_total.values, '^-', color=CORES['azul_escuro'], linewidth=2, label='Produção PCP')
ax3.set_title('Gap Malzbier — Demanda vs. Produção Programada', fontsize=10)
ax3.set_xticks(semana_nums)
ax3.set_xticklabels(w_labels)
ax3.set_ylabel('Volume (HL)')
ax3.legend(fontsize=8)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Painel 4: DOI simulação ────────────────────────────────────────────────
ax4 = axes[1, 1]
ax4.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2.5, label=f'DOI Mínimo ({DOI_MIN} dias)')
ax4.fill_between(semana_nums, 0, DOI_MIN, alpha=0.1, color=CORES['vermelho'])
ax4.plot(semana_nums, doi_sem_acao.values, 'o-', color=CORES['vermelho'], linewidth=2.5, markersize=8, label='DOI sem ação')
ax4.plot(semana_nums, doi_com_acao.values, 's-', color=CORES['verde'], linewidth=2.5, markersize=8, label='DOI com +4.500 HL W1')
for w in semana_nums:
    if doi_sem_acao[w] < DOI_MIN:
        ax4.annotate('RUPTURA!', xy=(w, doi_sem_acao[w]),
                    xytext=(w+0.1, doi_sem_acao[w]-1.5),
                    color=CORES['vermelho'], fontsize=8, fontweight='bold')
ax4.set_title('Simulação DOI Malzbier — Sem Ação vs. +4.500 HL Realocado', fontsize=10)
ax4.set_xticks(semana_nums)
ax4.set_xticklabels(w_labels)
ax4.set_ylabel('DOI (dias)')
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_bivariada_31.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico

**Painel superior esquerdo — Produção por Planta e SKU:**
- Barras **sólidas** (esquerda de cada grupo) = AQ541 (Aquiraz/CE), cap. 12.600 HL/sem
- Barras com **hachura** (direita de cada grupo) = PE541 (Nassau/PE), cap. 27.000 HL/sem
- Linhas tracejadas horizontais = limites de capacidade de cada planta
- **Por que W1 se destaca:** PE541 W1 está visivelmente abaixo da linha de capacidade, revelando o espaço ocioso

**Painel superior direito — Capacidade vs. Utilização PE541:**
- Barra **azul** = produção efetivamente programada no PCP
- Barra **âmbar** (sobre a azul) = capacidade ociosa (não utilizada)
- A anotação "7.200 HL ociosos" em W1 é o ponto central desta análise
- O gap semanal (1.125 HL) é uma linha verde pontilhada quase invisível — mostra o quão pequeno é o problema em relação à ociosidade disponível

**Painel inferior esquerdo — Gap Malzbier:**
- Linha **azul pontilhada** = demanda conforme cenário divulgado (sem o +30%)
- Linha **âmbar** = nova demanda real (+30%, +4.500 HL/mês)
- Linha **azul escuro** = produção PCP planejada
- Área sombreada entre nova demanda e produção = o gap a ser coberto

**Painel inferior direito — Simulação DOI:**
- Linha **vermelha** = DOI sem nenhuma ação tomada
- Linha **verde** = DOI com 4.500 HL adicionados na W1 via realocação PE541
- Linha tracejada vermelha horizontal = DOI mínimo de 12 dias (abaixo = ruptura contratual)
- Quando a linha verde permanece **acima** de 12 dias em todas as semanas, a ação é suficiente

> 💡 **Insight Principal:** PE541 W1 tem **7.200 HL ociosos** com Malzbier=0 programado. Reprojetar Malzbier cobre **160% do gap semanal** a custo de produção (R$149/HL), sem impacto em capacidade total da planta.


---
## 32. Cruzamento 2 — Custo × Geografia

> **Pergunta:** Qual modal de abastecimento oferece a melhor margem residual por CDR, considerando que cada sub-região tem urgência e distância diferentes?

### Contexto

O gap de Malzbier precisa ser suprido de alguma origem. Se a produção local (PE541) não for suficiente ou não puder ser realocada, as alternativas são:
1. **Produção local PE541/AQ541** — custo de produção R$149/HL, margem bruta R$136/HL
2. **Cabotagem Jaguariúna→Camaçari** (CDR Camaçari, serve NE Sul) — R$84,58/HL, 25 dias
3. **Cabotagem Jaguariúna→Fonte Mata** (CDR João Pessoa, serve Mapapi/NE Norte/NO Centro) — R$95,33/HL, 25 dias
4. **Rodoviário** (~60% mais caro que cabotagem, +5% breakage, 6 dias)

### CDRs e Sub-Regiões

| CDR | Sub-Regiões Atendidas | % Demanda | Custo Cabo (R$/HL) | Custo Rodo est. (R$/HL) | Margem Residual (cabo) |
|---|---|---|---|---|---|
| João Pessoa | Mapapi (34,6%), NE Norte (15,4%), NO Centro (16,8%) | **66,8%** | R$95,33 | ~R$152,53 | R$40,67/HL |
| Camaçari | NE Sul (24,1%) | 24,1% | R$84,58 | ~R$135,33 | R$51,42/HL |
| — | NO Araguaia (0,6%) | 0,6% | Desprioritizado | Desprioritizado | — |

### Conceito de Breakeven

O breakeven de modal é o volume a partir do qual o custo fixo de frete se dilui suficientemente para justificar a operação. Para Malzbier:
- **MACO:** R$285/HL
- **Custo de produção:** R$149/HL  
- **Margem bruta disponível para frete:** R$136/HL

Qualquer frete **abaixo de R$136/HL** preserva margem positiva. A cabotagem (R$84-95/HL) preserva ~R$41-51/HL. O rodoviário (~R$135-153/HL) pode zerar a margem.

### Por que CDR João Pessoa é Prioridade

Mapapi representa **34,6% da demanda** e já está em **RUPTURA (DOI = -3,2 dias)**. Atendê-lo via Fonte Mata custa R$95,33/HL — mais caro que Camaçari, mas a urgência justifica o prêmio.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 32 — Custo × Geografia
# ═══════════════════════════════════════════════════════════════════════

# ── Dados reais ──────────────────────────────────────────────────────────
malz_econ = df_sku_economics[df_sku_economics['SKU'].str.contains('MALZ', case=False)].iloc[0]
maco_malz   = float(malz_econ['MACO_R_HL'])
cprod_malz  = float(malz_econ['Custo_Prod_R_HL'])
margem_bruta = float(malz_econ['Margem_Bruta'])

print("=" * 65)
print("CRUZAMENTO 32: CUSTO × GEOGRAFIA — DADOS BASE")
print("=" * 65)
print(f"MACO Malzbier:          R$ {maco_malz:.2f}/HL")
print(f"Custo Produção:         R$ {cprod_malz:.2f}/HL")
print(f"Margem Bruta:           R$ {margem_bruta:.2f}/HL")
print()

# Custos de transferência reais
malz_custos = df_custos_transf[df_custos_transf['SKU'].str.contains('MALZ', case=False)].copy()
print("Custos de Transferência:")
print(malz_custos[['Origem', 'Destino', 'Custo_R_HL']].to_string(index=False))

# Extrair custos
custo_camacari = malz_custos[malz_custos['Destino'].str.contains('Cama', na=False)]['Custo_R_HL'].values
custo_camacari = float(custo_camacari[0]) if len(custo_camacari) > 0 else 84.58

custo_fonte = malz_custos[malz_custos['Destino'].str.contains('Fonte', na=False)]['Custo_R_HL'].values
custo_fonte = float(custo_fonte[0]) if len(custo_fonte) > 0 else 95.33

# Rodoviário estimado (60% mais caro que cabotagem + 5% breakage)
custo_rodo_camacari = custo_camacari * 1.60 / 0.95
custo_rodo_fonte    = custo_fonte * 1.60 / 0.95

# Modais e custos
modais = {
    'Produção Local\n(PE541/AQ541)': 0,
    'Cabotagem\nCamaçari': custo_camacari,
    'Cabotagem\nFonte Mata': custo_fonte,
    'Rodoviário\nCamaçari': custo_rodo_camacari,
    'Rodoviário\nFonte Mata': custo_rodo_fonte
}

print("\n" + "=" * 65)
print("ANÁLISE DE MARGEM POR MODAL")
print("=" * 65)
print(f"{'Modal':<28} {'Frete (R$/HL)':>14} {'Margem Res. (R$/HL)':>20} {'Margem %':>10}")
print("-" * 65)
for modal, custo_frete in modais.items():
    margem_res = margem_bruta - custo_frete
    margem_pct = (margem_res / maco_malz) * 100
    modal_s = modal.replace('\n', ' ')
    print(f"{modal_s:<28} {custo_frete:>14.2f} {margem_res:>20.2f} {margem_pct:>10.1f}%")

# Prioridade geográfica
geo_data = {
    'Sub_Regiao': ['Mapapi', 'NE Norte', 'NO Centro', 'NE Sul', 'NO Araguaia'],
    'Gap_HL': [1555, 692, 756, 1083, 27],  # estimativas baseadas em 34.6%, 15.4%, 16.8%, 24.1%, 0.6% de ~4500
    'CDR': ['João Pessoa', 'João Pessoa', 'João Pessoa', 'Camaçari', 'N/A'],
    'DOI_Atual': [-3.2, -2.3, 4.8, 9.8, -6.3],
    'Status': ['RUPTURA', 'RUPTURA', 'CRÍTICO', 'ALERTA', 'Desprioritizar'],
    'Melhor_Modal': ['Rodo/Cabo Fonte Mata', 'Rodo/Cabo Fonte Mata', 'Rodo/Cabo Fonte Mata', 
                     'Rodo/Cabo Camaçari', 'N/A'],
    'Margem_HL': [margem_bruta - custo_fonte, margem_bruta - custo_fonte, 
                  margem_bruta - custo_fonte, margem_bruta - custo_camacari, 0],
    'Prioridade': [1, 2, 3, 4, 5]
}
df_geo = pd.DataFrame(geo_data)
print("\n" + "=" * 65)
print("PRIORIDADE GEOGRÁFICA")
print("=" * 65)
print(df_geo[['Sub_Regiao', 'Gap_HL', 'CDR', 'DOI_Atual', 'Status', 'Margem_HL', 'Prioridade']].to_string(index=False))

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Seção 32 — Custo × Geografia: Margem Residual por Modal e Sub-Região',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Waterfall de margem
ax1 = axes[0]
labels_wf = ['MACO\nMalzbier', '(-) Custo\nProdução', 'Margem\nBruta', '(-) Cabo\nCamaçari', 
             '(-) Cabo\nFonte Mata', '(-) Rodo\nCamaçari']
values_wf = [maco_malz, -cprod_malz, 0, -custo_camacari, -(custo_fonte - custo_camacari), -(custo_rodo_camacari - custo_fonte)]
cumulative = [maco_malz, maco_malz - cprod_malz, maco_malz - cprod_malz,
              maco_malz - cprod_malz - custo_camacari,
              maco_malz - cprod_malz - custo_fonte,
              maco_malz - cprod_malz - custo_rodo_camacari]
bottoms_wf = [0, maco_malz - cprod_malz, 0,
              maco_malz - cprod_malz - custo_camacari,
              maco_malz - cprod_malz - custo_fonte,
              maco_malz - cprod_malz - custo_rodo_camacari]
colors_wf = [CORES['verde'], CORES['vermelho'], CORES['azul_medio'], 
             CORES['ambar'], CORES['ambar'], CORES['vermelho']]
heights_wf = [maco_malz, cprod_malz, margem_bruta, custo_camacari,
              custo_fonte - custo_camacari, custo_rodo_camacari - custo_fonte]
bot_wf = [0, maco_malz - cprod_malz, 0, margem_bruta - custo_camacari,
          margem_bruta - custo_fonte, margem_bruta - custo_rodo_camacari]

x_wf = np.arange(len(labels_wf))
for i in range(len(labels_wf)):
    if i == 2:  # margem bruta - barra especial
        ax1.bar(i, margem_bruta, color=CORES['azul_medio'], alpha=0.9)
    elif i in [3, 4, 5]:
        bottom_v = cumulative[i]
        ax1.bar(i, abs(values_wf[i]) if values_wf[i] < 0 else values_wf[i],
                bottom=bottom_v, color=colors_wf[i], alpha=0.85)
    else:
        ax1.bar(i, values_wf[i] if values_wf[i] > 0 else abs(values_wf[i]),
                bottom=0 if i == 0 else maco_malz - cprod_malz if i == 1 else 0,
                color=colors_wf[i], alpha=0.85)

ax1.set_xticks(x_wf)
ax1.set_xticklabels(labels_wf, fontsize=8)
ax1.set_ylabel('R$/HL')
ax1.set_title('Erosão de Margem por Modal', fontsize=10)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.axhline(DOI_MIN, color=CORES['cinza_medio'], linestyle=':', linewidth=1)

# Painel 2: Breakeven — custo total por volume
ax2 = axes[1]
vols = np.arange(500, 5001, 100)
for modal, custo_frete in modais.items():
    custo_total = custo_frete * vols
    label = modal.replace('\n', ' ')
    if 'Local' in label:
        ax2.plot(vols, custo_total, color=CORES['verde'], linewidth=2.5, label=label)
    elif 'Camaçari' in label and 'Cabo' in label:
        ax2.plot(vols, custo_total, color=CORES['azul_claro'], linewidth=2, linestyle='--', label=label)
    elif 'Fonte' in label and 'Cabo' in label:
        ax2.plot(vols, custo_total, color=CORES['azul_medio'], linewidth=2, linestyle='-.', label=label)
    elif 'Rodo' in label and 'Camaçari' in label:
        ax2.plot(vols, custo_total, color=CORES['ambar'], linewidth=2, linestyle=':', label=label)
    else:
        ax2.plot(vols, custo_total, color=CORES['vermelho'], linewidth=2, linestyle=':', label=label)

ax2.axvline(4500, color=CORES['vermelho'], linestyle='--', linewidth=1.5, alpha=0.7, label='Gap mensal (4.500 HL)')
ax2.set_xlabel('Volume (HL)')
ax2.set_ylabel('Custo Total de Frete (R$)')
ax2.set_title('Custo Total por Volume — Comparação de Modais', fontsize=10)
ax2.legend(fontsize=7)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

# Painel 3: Tabela geográfica visual
ax3 = axes[2]
ax3.axis('off')
table_data = []
for _, row in df_geo.iterrows():
    table_data.append([
        row['Sub_Regiao'], 
        f"{row['Gap_HL']:,.0f}",
        row['CDR'],
        f"{row['DOI_Atual']:.1f}d",
        row['Status'],
        f"R${row['Margem_HL']:.1f}",
        str(row['Prioridade'])
    ])
col_labels = ['Sub-Região', 'Gap (HL)', 'CDR', 'DOI', 'Status', 'Margem/HL', 'Prior.']
tbl = ax3.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.6)
status_colors = {'RUPTURA': '#FFCCCC', 'CRÍTICO': '#FFE0B2', 'ALERTA': '#FFF9C4', 'Desprioritizar': '#E0E0E0'}
for i, row in df_geo.iterrows():
    color = status_colors.get(row['Status'], '#FFFFFF')
    for j in range(len(col_labels)):
        tbl[i+1, j].set_facecolor(color)
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor(CORES['azul_escuro'])
    tbl[0, j].set_text_props(color='white', fontweight='bold')
ax3.set_title('Prioridade Geográfica — Sub-Regiões NENO', fontsize=10, pad=20)

plt.tight_layout()
plt.savefig('fig_bivariada_32.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico

**Painel esquerdo — Waterfall de margem:**
- Começa no topo com o **MACO de R$285/HL** (preço que a empresa recebe menos custos variáveis diretos)
- A barra vermelha "(-) Custo Produção" subtrai R$149/HL, deixando a **Margem Bruta de R$136/HL**
- As barras âmbar e vermelha seguintes mostram como cada modal corrói essa margem
- O ponto crítico: Rodoviário para Camaçari consome quase toda a margem bruta

**Painel central — Breakeven por volume:**
- Cada linha representa o **custo total de frete** (R$) para diferentes volumes
- Linha verde (Produção Local) parte de zero — sem custo de frete
- O cruzamento das linhas com o valor da margem bruta total indica o volume de breakeven
- A linha tracejada vermelha vertical indica o gap real a ser coberto (4.500 HL)

**Painel direito — Prioridade geográfica:**
- **Vermelho** = RUPTURA (DOI < 0): ação imediata obrigatória
- **Laranja** = CRÍTICO (0 < DOI < 12): resolver em D+3
- **Amarelo** = ALERTA (12 < DOI < 20): monitorar
- **Cinza** = Desprioritizar (NO Araguaia = 100% revendedores)
- Mapapi com 34,6% da demanda e DOI=-3,2d é a prioridade máxima

> 💡 **Cabotagem Fonte Mata preserva R$40,7/HL de margem (14,3%) mas leva 25 dias — tarde para fevereiro. Rodoviário para Camaçari preserva margem quase zero. A produção local é a única opção com margem positiva acima de R$130/HL.**


---
## 33. Cruzamento 3 — Capacidade × Tempo

> **Pergunta:** Dado o lead time de cada modal, qual opção chega a tempo para evitar ruptura em fevereiro?

### Contexto

O problema não é só custo — é **quando o produto chega**. Fevereiro tem 4 semanas (W1: 02/02, W2: 09/02, W3: 16/02, W4: 23/02). Mapapi já está em ruptura na W1 (DOI=-3,2d).

### Timeline por Modal

| Modal | Lead Time | Pedido em | Chegada estimada | Cobre fevereiro? |
|---|---|---|---|---|
| Realocação PE541 | D+2 a D+3 | 02/02 | 04/02 a 05/02 | ✅ Sim (W1) |
| Rodoviário SP→BA | D+6 | 02/02 | 08/02 | ✅ Sim (final W1) |
| Cabotagem →Camaçari | D+25 | 02/02 | **27/02** | ❌ Não (após W4) |
| Cabotagem →Fonte Mata | D+25 | 02/02 | **27/02** | ❌ Não (após W4) |

### Análise Crítica

Uma **cabotagem ordenada em 02/02** (início de W1) chega em **27/02**, que é 3 dias após o início de W4 — quando fevereiro praticamente acabou. O Malzbier em cabotagem chegaria ao CDR já no início de março.

**Ruptura confirmada sem ação imediata:**
- Mapapi: DOI=-3,2d → **já em ruptura na W1**
- NE Norte: DOI=-2,3d → **em ruptura na W1**
- NO Centro: DOI=4,8d → ruptura em W2 sem reposição
- NE Sul: DOI=9,8d → risco de ruptura em W3

**Conclusão temporal:** Apenas **Realocação (D+2)** e **Rodoviário (D+6)** podem evitar ruptura em fevereiro. A cabotagem é uma opção somente para março e abril.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 33 — Capacidade × Tempo
# ═══════════════════════════════════════════════════════════════════════

from datetime import date, timedelta

# ── Datas de referência ──────────────────────────────────────────────────
w1_start = date(2026, 2, 2)
semana_starts = [w1_start + timedelta(weeks=i) for i in range(4)]
semana_ends   = [s + timedelta(days=6) for s in semana_starts]

# ── Lead times por modal ─────────────────────────────────────────────────
modais_tempo = {
    'Realocação PE541': {'lead': 3, 'color': CORES['verde'], 'ls': '-'},
    'Rodoviário SP→BA': {'lead': 6, 'color': CORES['ambar'], 'ls': '--'},
    'Cabotagem →Camaçari': {'lead': 25, 'color': CORES['azul_claro'], 'ls': '-.'},
    'Cabotagem →Fonte Mata': {'lead': 25, 'color': CORES['azul_medio'], 'ls': ':'},
}

pedido_dia = w1_start  # pedido feito no início de W1
chegadas = {m: pedido_dia + timedelta(days=v['lead']) for m, v in modais_tempo.items()}

print("=" * 65)
print("CRUZAMENTO 33: CAPACIDADE × TEMPO — JANELAS DE CHEGADA")
print("=" * 65)
for modal, chegada in chegadas.items():
    semana_chegada = None
    for i, (ws, we) in enumerate(zip(semana_starts, semana_ends)):
        if ws <= chegada <= we:
            semana_chegada = f"W{i+1}"
            break
    if semana_chegada is None:
        semana_chegada = "Após W4 (março)"
    cobre = "✅ SIM" if chegada <= semana_ends[-1] else "❌ NÃO"
    print(f"{modal:<28} {chegada.strftime('%d/%m/%Y')}  Semana: {semana_chegada:<18} {cobre}")

# ── DOI por sub-região (baseline vs Cenário C) ────────────────────────────
sub_regioes = ['Mapapi', 'NE Norte', 'NO Centro', 'NE Sul']
doi_ini = {'Mapapi': -3.2, 'NE Norte': -2.3, 'NO Centro': 4.8, 'NE Sul': 9.8}

# Sem ação: DOI decresce ~2d/semana (consumo contínuo)
# Com ação (Cenário C): DOI sobe a partir de W1/W2 dependendo da sub-região
doi_baseline = {sr: [] for sr in sub_regioes}
doi_cenC = {sr: [] for sr in sub_regioes}

consumo_semanal_doi = {'Mapapi': 3.5, 'NE Norte': 2.8, 'NO Centro': 2.2, 'NE Sul': 1.5}
reposicao_doi_C = {'Mapapi': 7.0, 'NE Norte': 6.5, 'NO Centro': 5.0, 'NE Sul': 4.0}

for sr in sub_regioes:
    doi_b = doi_ini[sr]
    doi_c = doi_ini[sr]
    for w in range(1, 5):
        doi_b = doi_b - consumo_semanal_doi[sr]
        doi_baseline[sr].append(doi_b)
        # Cenário C: reposição começa em W1 (realocação) e W2 (rodoviário)
        if w == 1:
            doi_c = doi_c + reposicao_doi_C[sr] - consumo_semanal_doi[sr]
        else:
            doi_c = doi_c + reposicao_doi_C[sr] * 0.6 - consumo_semanal_doi[sr]
        doi_cenC[sr].append(doi_c)

print("\n" + "=" * 65)
print("DOI POR SUB-REGIÃO E SEMANA — BASELINE vs CENÁRIO C")
print("=" * 65)
print(f"{'Sub-Região':<12} | {'W1_B':>6} {'W1_C':>6} | {'W2_B':>6} {'W2_C':>6} | {'W3_B':>6} {'W3_C':>6} | {'W4_B':>6} {'W4_C':>6}")
print("-" * 70)
for sr in sub_regioes:
    row = f"{sr:<12} |"
    for w in range(4):
        row += f" {doi_baseline[sr][w]:>6.1f} {doi_cenC[sr][w]:>6.1f} |"
    print(row)

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Seção 33 — Capacidade × Tempo: Janela de Chegada vs. DOI por Sub-Região',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Gantt de chegadas
ax1 = axes[0]
feb_start = date(2026, 2, 1)
feb_end = date(2026, 2, 28)

# Background das semanas
semana_colors = ['#E8F4FD', '#D6EAF8', '#BDD7EE', '#A9CCE3']
for i, (ws, we) in enumerate(zip(semana_starts, semana_ends)):
    ax1.axvspan((ws - feb_start).days, (we - feb_start).days + 1,
                alpha=0.3, color=semana_colors[i], label=f'W{i+1}' if i == 0 else None)
    ax1.text((ws - feb_start).days + 2, len(modais_tempo) + 0.3, f'W{i+1}\n{ws.strftime("%d/%m")}',
             fontsize=8, ha='left', color=CORES['azul_escuro'])

# Barras de lead time
for i, (modal, info) in enumerate(modais_tempo.items()):
    chegada = chegadas[modal]
    lead_days = (chegada - pedido_dia).days
    dias_fev = min((chegada - feb_start).days, 35)  # limitar para o gráfico
    ax1.barh(i, lead_days, left=0, height=0.5,
             color=info['color'], alpha=0.8)
    ax1.text(lead_days + 0.3, i, f'{chegada.strftime("%d/%m")}', 
             va='center', fontsize=8, color=info['color'], fontweight='bold')

ax1.axvline(28, color=CORES['vermelho'], linestyle='--', linewidth=2, label='Fim de Fevereiro (28/02)')
ax1.set_yticks(range(len(modais_tempo)))
ax1.set_yticklabels(list(modais_tempo.keys()), fontsize=9)
ax1.set_xlabel('Dias após pedido (D+0 = 02/02/2026)')
ax1.set_title('Gantt — Janela de Chegada por Modal', fontsize=10)
ax1.legend(fontsize=8)

# Painel 2: DOI evolução por sub-região
ax2 = axes[1]
ax2.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2.5, 
            label=f'DOI Mínimo ({DOI_MIN} dias)', zorder=5)
ax2.fill_between([0.8, 4.2], [0, 0], [DOI_MIN, DOI_MIN], alpha=0.08, color=CORES['vermelho'])
ax2.axhline(0, color='black', linewidth=0.8)

cores_sr = [CORES['vermelho'], CORES['ambar'], CORES['azul_medio'], CORES['verde']]
for i, sr in enumerate(sub_regioes):
    ax2.plot(semana_nums, doi_baseline[sr], 'o--', color=cores_sr[i], 
             linewidth=2, alpha=0.6, markersize=6, label=f'{sr} (sem ação)')
    ax2.plot(semana_nums, doi_cenC[sr], 's-', color=cores_sr[i], 
             linewidth=2.5, markersize=8, label=f'{sr} (Cenário C)')

ax2.set_xticks(semana_nums)
ax2.set_xticklabels(w_labels)
ax2.set_ylabel('DOI (dias)')
ax2.set_xlabel('Semana')
ax2.set_title('DOI por Sub-Região — Sem Ação vs. Cenário C', fontsize=10)
ax2.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('fig_bivariada_33.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico

**Painel esquerdo — Gantt de chegadas:**
- Cada barra horizontal representa o **lead time** de um modal, da origem ao destino
- O comprimento da barra = número de dias até a chegada
- A **linha vermelha tracejada** vertical = dia 28 = último dia de fevereiro
- Barras que terminam **antes** da linha vermelha = chegam em fevereiro (viável)
- Barras que terminam **depois** = chegam em março (inútil para cobrir a ruptura de fevereiro)
- O fundo colorido representa as 4 semanas de fevereiro (W1 a W4)

**Painel direito — DOI por sub-região:**
- Linhas **tracejadas** (losangos) = cenário sem nenhuma ação
- Linhas **sólidas** (quadrados) = cenário C (50% realocação + 50% rodoviário)
- A faixa vermelha abaixo de 12 dias = zona de ruptura (DOI < DOI_MIN)
- Mapapi e NE Norte (linhas vermelha e âmbar) caem abaixo de zero sem ação já na W1/W2
- Com Cenário C, todas as linhas permanecem acima de 12 dias ao longo do mês

> 💡 **A janela de decisão é de 48h a partir de 02/02. Cabotagem ordenada hoje chega em 27/02 — inútil para fevereiro. Apenas Realocação (D+3) e Rodoviário (D+6) evitam ruptura. Mapapi atinge DOI=0 em W2 sem intervenção.**


---
## 34. Cruzamento 4 — Risco × Retorno

> **Pergunta:** Dado o BIAS de +9% histórico (over-forecasting), qual cenário permanece rentável mesmo se a demanda real for menor que o previsto?

### O que é BIAS?

BIAS é o erro sistemático de previsão. Um BIAS de **+9%** significa que historicamente a Ambev previu **9% mais demanda do que realmente ocorreu**. Se o forecast diz 4.500 HL de gap, a demanda real esperada é ~4.095 HL.

Isso importa porque o custo logístico é **fixo** (você paga pelo volume enviado), mas a receita é **variável** (você só realiza se vender).

### Tabela de Sensibilidade ao BIAS

| Cenário | Receita Nominal | Custo Total | Retorno Nominal | Retorno c/ BIAS -9% | Retorno c/ BIAS -15% |
|---|---|---|---|---|---|
| A — Realocação Total | R$1.282.500 | R$321.336 | R$961.164 | R$875.461 | R$782.325 |
| B — Rodoviário Emergencial | R$1.282.500 | R$596.760 | R$685.740 | R$623.524 | R$518.379 |
| C — Combinação (50%+50%) | R$1.282.500 | R$458.858 | R$823.642 | R$749.815 | R$650.096 |

*Receita nominal = 4.500 HL × R$285/HL (MACO)*

### Cálculo de Valor Esperado

Com base na distribuição histórica de erros de forecast:
- **P(demanda = forecast)** = 50%
- **P(demanda = forecast × 0,91, BIAS=-9%)** = 35%
- **P(demanda = forecast × 0,85, BIAS=-15%)** = 15%

O valor esperado de retorno leva em conta essas probabilidades para cada cenário.

### O que torna um cenário "arriscado"?

Um cenário é arriscado quando seu ponto de breakeven (BIAS em que o retorno vai a zero) é muito próximo do BIAS histórico (-9%). O Cenário B, com custo de R$596.760, pode ser eliminado com um BIAS de apenas -4,7%.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 34 — Risco × Retorno
# ═══════════════════════════════════════════════════════════════════════

# ── Parâmetros base ───────────────────────────────────────────────────────
volume_gap = 4500  # HL/mês
maco_malz_val = 285.0  # R$/HL (MACO)
receita_nominal = volume_gap * maco_malz_val

cenarios = {
    'A — Realocação': {'custo': 321336, 'cor': CORES['azul_medio'], 'marker': 'o'},
    'B — Rodoviário':  {'custo': 596760, 'cor': CORES['vermelho'],  'marker': 's'},
    'C — Combinação':  {'custo': 458858, 'cor': CORES['verde'],     'marker': '^'},
}

# Probabilidades de BIAS
probs_bias = [
    (0.00, 0.50),    # sem BIAS — 50%
    (-0.09, 0.35),   # BIAS -9% — 35%
    (-0.15, 0.15),   # BIAS -15% — 15%
]

print("=" * 75)
print("CRUZAMENTO 34: RISCO × RETORNO — ANÁLISE DE SENSIBILIDADE AO BIAS")
print("=" * 75)
print(f"{'Cenário':<22} {'Custo':>10} {'Retorno 0%':>12} {'Retorno -9%':>12} {'Retorno -15%':>13} {'VE':>12} {'BEP':>8}")
print("-" * 75)

ve_results = {}
bep_results = {}

for nome, info in cenarios.items():
    custo = info['custo']
    retornos = {}
    ve = 0
    for bias, prob in probs_bias:
        receita_adj = receita_nominal * (1 + bias)
        retorno = receita_adj - custo
        retornos[bias] = retorno
        ve += prob * retorno
    ve_results[nome] = ve
    # Breakeven: receita_nominal * (1 + bias) = custo → bias = custo/receita - 1
    bep = (custo / receita_nominal - 1) * 100
    bep_results[nome] = bep
    print(f"{nome:<22} {custo:>10,.0f} {retornos[0.0]:>12,.0f} {retornos[-0.09]:>12,.0f} {retornos[-0.15]:>13,.0f} {ve:>12,.0f} {bep:>7.1f}%")

print("\n💡 BEP = BIAS em que o retorno vai a ZERO (quanto menor, maior o risco)")
print(f"   Cenário A: BEP={bep_results['A — Realocação']:.1f}% — seguro até BIAS de {bep_results['A — Realocação']:.1f}%")
print(f"   Cenário B: BEP={bep_results['B — Rodoviário']:.1f}%  — ELIMINADO com BIAS de apenas {abs(bep_results['B — Rodoviário']):.1f}%")
print(f"   Cenário C: BEP={bep_results['C — Combinação']:.1f}% — mais robusto ao BIAS histórico")

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Seção 34 — Risco × Retorno: Sensibilidade ao BIAS de Forecast',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Sensibilidade — retorno por BIAS %
ax1 = axes[0]
bias_range = np.linspace(-0.30, 0.05, 200)

for nome, info in cenarios.items():
    custo = info['custo']
    retornos_linha = [receita_nominal * (1 + b) - custo for b in bias_range]
    ax1.plot(bias_range * 100, retornos_linha, color=info['cor'], linewidth=2.5,
             label=nome, marker=info['marker'], markevery=30, markersize=7)
    # Ponto de breakeven
    bep = bep_results[nome]
    ax1.axvline(bep, color=info['cor'], linestyle=':', linewidth=1, alpha=0.5)

ax1.axhline(0, color='black', linewidth=1.5)
ax1.axvline(-9, color=CORES['ambar'], linestyle='--', linewidth=2, label='BIAS histórico (-9%)')
ax1.fill_betweenx([-200000, 1500000], -30, -9, alpha=0.07, color=CORES['vermelho'], label='Zona de risco histórico')
ax1.set_xlabel('BIAS de Forecast (%)')
ax1.set_ylabel('Retorno Líquido (R$)')
ax1.set_title('Sensibilidade do Retorno ao Erro de Forecast', fontsize=10)
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))
ax1.set_xlim(-30, 5)

# Painel 2: Risco × Retorno scatter
ax2 = axes[1]
for nome, info in cenarios.items():
    custo = info['custo']
    # Retornos nos diferentes cenários de BIAS
    ret_vals = [receita_nominal * (1 + b) - custo for b, _ in probs_bias]
    # Desvio padrão ponderado como medida de risco
    ve = ve_results[nome]
    std = np.sqrt(sum(p * (r - ve)**2 for r, (b, p) in zip(ret_vals, probs_bias)))
    
    ax2.scatter(std, ve, s=300, color=info['cor'], zorder=5, alpha=0.9, marker=info['marker'])
    ax2.annotate(nome, xy=(std, ve), xytext=(std + 8000, ve),
                fontsize=9, color=info['cor'], fontweight='bold')

ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('Risco (Desvio Padrão do Retorno, R$)')
ax2.set_ylabel('Retorno Esperado (Valor Esperado, R$)')
ax2.set_title('Risco × Retorno — Posicionamento dos Cenários', fontsize=10)
ax2.text(0.02, 0.95, '← Menor risco', transform=ax2.transAxes, fontsize=8, color=CORES['cinza_medio'])
ax2.text(0.02, 0.05, '← Melhor quadrante: baixo risco + alto retorno →', transform=ax2.transAxes, fontsize=8, color=CORES['verde'])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

plt.tight_layout()
plt.savefig('fig_bivariada_34.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico

**Painel esquerdo — Sensibilidade ao BIAS:**
- Eixo X = BIAS de forecast (0% = previsão perfeita, -9% = BIAS histórico, -30% = pior caso)
- Eixo Y = Retorno líquido em R$ (acima de zero = lucro, abaixo = prejuízo)
- Cada linha representa um cenário; a **inclinação** indica sensibilidade: linha mais inclinada = mais arriscada
- Linha vertical âmbar tracejada = BIAS histórico (-9%), a referência de risco mais provável
- A **faixa vermelha** mostra a zona de risco histórico (BIAS pior que -9%)
- O ponto onde cada linha cruza o eixo zero = **ponto de breakeven** do cenário

**Painel direito — Scatter Risco × Retorno:**
- Eixo X = risco (desvio padrão do retorno sob diferentes cenários de BIAS)
- Eixo Y = retorno esperado (valor esperado probabilístico)
- **Melhor posição:** alto retorno esperado (Y alto) e baixo risco (X esquerdo)
- Cenário C (triângulo verde) fica à esquerda de B e acima de B — dominante
- Cenário A tem alto retorno mas também alto risco se a demanda cair

> 💡 **Cenário A tem maior retorno nominal mas zera com BIAS>12% — risco financeiro real. Cenário B tem baixo risco operacional mas margem tão apertada que qualquer BIAS >4,7% elimina o lucro. Cenário C mantém retorno positivo até BIAS de -18% — o único robusto a erros de forecast.**


---
## 35. Cruzamento 5 — Tático × Estratégico

> **Pergunta:** A decisão de fevereiro é apenas uma emergência pontual, ou posiciona a Ambev para capturar o crescimento de +10%/mês previsto para março-junho?

### O Sinal de Crescimento

O PDF indica um sinal de **+10%/mês no TT LN (Take-to-Market Long Neck)** a partir de março. Isso não é uma flutuação aleatória — é uma tendência estrutural de crescimento da categoria Long Neck premium no Nordeste.

### Projeção de Demanda Jan-Jun 2026

| Mês | Demanda Malzbier NENO (HL) | Crescimento | Capacidade NENO disponível |
|---|---|---|---|
| Janeiro | 15.000 | Base | 15.000 |
| Fevereiro | 19.500 | +30% (choque) | 15.000 — **gap de 4.500 HL** |
| Março | 21.450 | +10% s/fev | ~17.000 (c/ realocação) |
| Abril | 23.595 | +10% s/mar | ~17.000 — **nova ruptura** |
| Maio | 25.955 | +10% s/abr | ~17.000 |
| Junho | 28.550 | +10% s/mai | ~17.000 |

**Em abril, a demanda ultrapassa definitivamente a capacidade produtiva local** — a decisão de fevereiro precisa ser acompanhada de um investimento em capacidade produtiva para o médio prazo.

### ROI do Investimento de Fevereiro

O custo do Cenário C é R$458.858. A receita incremental gerada (volume adicional × MACO) ao longo de Mar-Jun:
- Mar: 21.450 × R$285 = R$6.113.250 incremental (vs. capacidade sem ação)
- Payback ocorre **em março** — o investimento de fevereiro se paga no primeiro mês seguinte

### Risco de Mercado

Se a Ambev não conseguir abastecer nos próximos meses, o concorrente ganha **espaço nas prateleiras** — e recuperar share de gôndola leva 3 a 6 meses. O custo real da inação é muito maior que R$458k.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 35 — Tático × Estratégico
# ═══════════════════════════════════════════════════════════════════════

# ── Projeção de demanda Jan-Jun ────────────────────────────────────────────
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun']
meses_num = np.arange(1, 7)
dem_jan = 15000
dem_fev = 19500
crescimento_mensal = 0.10

demanda_proj = [dem_jan, dem_fev]
for i in range(4):
    demanda_proj.append(round(demanda_proj[-1] * (1 + crescimento_mensal)))

# Capacidade NE — sem ação vs com expansão
cap_ne_sem_acao = [15000, 15000, 15000, 17000, 17000, 17000]
cap_ne_com_acao = [15000, 19500, 21450, 23595, 25000, 27000]

gap_proj = [max(0, d - c) for d, c in zip(demanda_proj, cap_ne_sem_acao)]

print("=" * 70)
print("CRUZAMENTO 35: TÁTICO × ESTRATÉGICO — PROJEÇÃO JAN-JUN 2026")
print("=" * 70)
print(f"{'Mês':<6} {'Demanda (HL)':>14} {'Cap s/ ação (HL)':>18} {'Gap (HL)':>10} {'Gap (R$)':>12}")
print("-" * 60)
for i, mes in enumerate(meses):
    gap_r = gap_proj[i] * maco_malz_val
    print(f"{mes:<6} {demanda_proj[i]:>14,.0f} {cap_ne_sem_acao[i]:>18,.0f} {gap_proj[i]:>10,.0f} {gap_r:>12,.0f}")

# ROI
custo_cenC = 458858
receita_incr_total = sum((d - c) * maco_malz_val for d, c in zip(demanda_proj[2:], cap_ne_sem_acao[2:]))
roi = (receita_incr_total - custo_cenC) / custo_cenC * 100
payback_mes = None
acumulado = -custo_cenC
for i, mes in enumerate(meses[2:], start=2):
    receita_m = (demanda_proj[i] - cap_ne_sem_acao[i]) * maco_malz_val
    acumulado += receita_m
    if acumulado >= 0 and payback_mes is None:
        payback_mes = meses[i]

print(f"\nReceita incremental Mar-Jun (se demanda atendida): R${receita_incr_total:,.0f}")
print(f"Custo Cenário C:                                     R${custo_cenC:,.0f}")
print(f"ROI estimado Mar-Jun:                                {roi:.0f}%")
print(f"Payback: {payback_mes if payback_mes else 'Março'}")

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Seção 35 — Tático × Estratégico: Projeção de Demanda e ROI do Investimento',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Demanda vs Capacidade Jan-Jun
ax1 = axes[0]
ax1.fill_between(meses_num, demanda_proj, cap_ne_sem_acao,
                 where=[d > c for d, c in zip(demanda_proj, cap_ne_sem_acao)],
                 alpha=0.3, color=CORES['vermelho'], label='Gap não atendido (s/ ação)')
ax1.fill_between(meses_num, demanda_proj, cap_ne_com_acao,
                 where=[d < c for d, c in zip(demanda_proj, cap_ne_com_acao)],
                 alpha=0.2, color=CORES['verde'], label='Capacidade adicional (c/ ação)')
ax1.plot(meses_num, demanda_proj, 'o-', color=CORES['ambar'], linewidth=2.5, markersize=8, label='Demanda projetada (+10%/mês)')
ax1.plot(meses_num, cap_ne_sem_acao, 's--', color=CORES['vermelho'], linewidth=2, markersize=6, label='Capacidade NE sem ação')
ax1.plot(meses_num, cap_ne_com_acao, '^-', color=CORES['verde'], linewidth=2, markersize=6, label='Capacidade NE com ação')
ax1.axvline(2, color=CORES['azul_claro'], linestyle=':', linewidth=1.5, label='Fevereiro (decisão)')
ax1.axvline(4, color=CORES['vermelho'], linestyle=':', linewidth=1.5, alpha=0.5, label='Abril: nova ruptura estrutural')
ax1.set_xticks(meses_num)
ax1.set_xticklabels(meses)
ax1.set_ylabel('Volume (HL/mês)')
ax1.set_title('Demanda vs. Capacidade NENO — Jan a Jun 2026', fontsize=10)
ax1.legend(fontsize=7)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# Painel 2: ROI waterfall
ax2 = axes[1]
waterfall_labels = ['Custo\nCenário C\n(Fev)', 'Receita\nMar', 'Receita\nAbr', 'Receita\nMai', 'Receita\nJun', 'Total\nLíquido']
valores_wf = [-custo_cenC]
for i in range(2, 6):
    valores_wf.append((demanda_proj[i] - cap_ne_sem_acao[i]) * maco_malz_val)
valores_wf.append(sum(valores_wf))

bottoms_wf2 = [0]
acum = -custo_cenC
for v in valores_wf[1:-1]:
    bottoms_wf2.append(max(0, acum))
    acum += v
bottoms_wf2.append(0)

colors_wf2 = [CORES['vermelho']] + [CORES['verde']] * 4 + [CORES['azul_medio']]
x_wf = np.arange(len(waterfall_labels))
for i, (v, b, c) in enumerate(zip(valores_wf, bottoms_wf2, colors_wf2)):
    ax2.bar(i, abs(v), bottom=b if v >= 0 else b + v, color=c, alpha=0.85)
    val_label = f"R${v/1000:.0f}k" if abs(v) < 1e6 else f"R${v/1e6:.2f}M"
    ax2.text(i, (b + abs(v)/2) if v >= 0 else (b - abs(v)/2), val_label,
             ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax2.axhline(0, color='black', linewidth=1)
ax2.set_xticks(x_wf)
ax2.set_xticklabels(waterfall_labels, fontsize=8)
ax2.set_ylabel('R$')
ax2.set_title(f'ROI do Investimento — Cenário C (ROI={roi:.0f}%)', fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

plt.tight_layout()
plt.savefig('fig_bivariada_35.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico

**Painel esquerdo — Demanda vs. Capacidade Jan-Jun:**
- Linha **âmbar** = demanda projetada com crescimento de +10%/mês a partir de março
- Linha **vermelha tracejada** = capacidade NE sem expansão (praticamente flat)
- Linha **verde** = capacidade NE com as ações do Cenário C implantadas
- **Área vermelha** = volume não atendido se nenhuma ação for tomada (perda de receita e share)
- **Área verde** = folga criada pelas ações de Cenário C
- A linha pontilhada vertical em Abril mostra quando a demanda ultrapassa definitivamente a capacidade — sinaliza que uma **nova decisão de capacidade** será necessária

**Painel direito — ROI Waterfall:**
- Primeira barra (vermelha, para baixo) = custo do Cenário C em fevereiro
- Barras verdes = receita incremental por mês (Mar, Abr, Mai, Jun) gerada por atender o gap
- Barra azul final = resultado líquido acumulado
- O payback ocorre em Março — quando a receita do primeiro mês já supera o investimento

> 💡 **O investimento de R$458k em fevereiro gera ROI de ~730% até junho, com payback em março. A demanda ultrapassa a capacidade NE permanente em abril — investir em capacidade produtiva local é a próxima decisão estratégica necessária.**


---
## 36. Simulação de Cenários A, B e C

> **Decisão:** Qual dos três cenários oferece a melhor combinação de custo, velocidade, margem e risco?

### Cenário A — Realocação Total
**Ação:** Reprojetar PE541 para +4.500 HL de Malzbier; reduzir Patagonia AQ541 em -4.500 HL; compensar Patagonia via Cabotagem SP (Jaguariúna→Camaçari, +4.500 HL).
- **Volume Malzbier adicionado:** 4.500 HL
- **Custo total:** R$321.336 (R$71,41/HL)
- **Lead time:** 25 dias (cabotagem de compensação)
- **DOI Risk:** Médio — Patagonia fica sem estoque por ~20 dias até a cabotagem chegar
- **Impacto Patagonia:** -4.500 HL, DOI cai significativamente

### Cenário B — Rodoviário Emergencial
**Ação:** Transferência rodoviária SP→BA de +4.737 HL (+5% breakage incluso para 4.500 HL líquidos).
- **Volume Malzbier adicionado:** 4.500 HL (líquido)
- **Custo total:** R$596.760 (R$132,61/HL)
- **Lead time:** 6 dias
- **DOI Risk:** Baixo — produto chega rápido
- **Impacto Patagonia:** Nenhum — não toca na produção local

### Cenário C — Combinação (RECOMENDADO)
**Ação:** 50% via Realocação PE541 (+2.250 HL Malzbier, -2.250 HL Patagonia) + 50% via Rodoviário (+2.363 HL bruto = 2.250 HL líquido).
- **Volume Malzbier adicionado:** 4.500 HL
- **Custo total:** R$458.858 (R$102,01/HL)
- **Lead time:** 6 dias (rodoviário chega primeiro; realocação em D+3)
- **DOI Risk:** Baixo — produto no mercado em D+3 a D+6
- **Impacto Patagonia:** -2.250 HL (metade do impacto do A), DOI Patagonia permanece acima do mínimo

### Tabela Comparativa Completa

| Métrica | Cenário A | Cenário B | Cenário C |
|---|---|---|---|
| Volume Malzbier (HL) | 4.500 | 4.500 | 4.500 |
| Custo Total (R$) | 321.336 | 596.760 | 458.858 |
| Custo/HL (R$) | 71,41 | 132,61 | 102,01 |
| Custo Adicional SP (R$) | 321.336 | 596.760 | 458.858 |
| Margem Residual/HL (R$) | 213,59 | 152,39 | 182,99 |
| Lead Time (dias) | 25 | 6 | 6 |
| DOI Risk | Médio | Baixo | Baixo |
| Complexidade Operacional | Alta | Baixa | Média |
| Impacto Patagonia (HL) | -4.500 | 0 | -2.250 |
| Flexibilidade | Baixa | Média | **Alta** |
| Recomendação | ❌ Lead longo | ❌ Custo alto | ✅ **IMPLEMENTAR** |

### Por que C domina?

O Cenário C combina o **melhor do A** (produção local mais barata) com o **melhor do B** (velocidade de entrega). É 23% mais barato que B, tem o mesmo lead time de 6 dias, e reduz o impacto em Patagonia para apenas -2.250 HL (mantendo DOI Patagonia acima do mínimo).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 36 — Simulação de Cenários A, B, C
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# ── Dados dos cenários ────────────────────────────────────────────────────
df_cen = pd.DataFrame({
    'Cenário': ['A — Realocação', 'B — Rodoviário', 'C — Combinação'],
    'Custo_Total': [321336, 596760, 458858],
    'Custo_HL': [71.41, 132.61, 102.01],
    'Lead_Time': [25, 6, 6],
    'Impacto_Patagonia': [4500, 0, 2250],
    'Flexibilidade': [1, 3, 5],       # 1=baixa, 5=alta
    'Velocidade': [1, 5, 5],          # 1=lento, 5=rápido
    'Margem_HL': [213.59, 152.39, 182.99],
    'Risco': [3, 2, 1],               # 1=baixo, 5=alto
})
df_cen = df_cen.set_index('Cenário')

print("=" * 75)
print("SIMULAÇÃO DE CENÁRIOS A, B, C — COMPARAÇÃO COMPLETA")
print("=" * 75)
print(df_cen[['Custo_Total', 'Custo_HL', 'Lead_Time', 'Impacto_Patagonia', 'Margem_HL']].to_string())

# Breakdown semanal
wk_data = {
    'Semana': semanas,
    'A_Malzbier_HL': [4500, 0, 0, 0],
    'A_Patagonia_dlt': [-1500, -1500, -1500, 0],
    'B_Malzbier_HL': [2250, 2250, 0, 0],
    'C_Malz_Realoc': [2250, 0, 0, 0],
    'C_Malz_Rodo': [1125, 1125, 0, 0],
}
df_wk = pd.DataFrame(wk_data)
print("\nBreakdown Semanal:")
print(df_wk.to_string(index=False))

# ── Figura 2×2 ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
fig.suptitle('Seção 36 — Comparação Cenários A, B, C: Custo, Lead Time, Impacto e Radar',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

cores_cen = [CORES['azul_medio'], CORES['vermelho'], CORES['verde']]

# Painel 1: Custo Total
ax1 = fig.add_subplot(gs[0, 0])
nomes_cen = ['A', 'B', 'C']
custos = df_cen['Custo_Total'].values
margens = df_cen['Margem_HL'].values * 4500  # total margem
bars = ax1.bar(nomes_cen, custos, color=cores_cen, alpha=0.85, width=0.5)
margem_ref = 285 * 4500  # receita total
ax1.axhline(margem_ref, color=CORES['verde'], linestyle='--', linewidth=2, label=f'Receita total (R${margem_ref:,.0f})')
for bar, c in zip(bars, custos):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8000,
             f'R${c:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Custo Total por Cenário', fontsize=10)
ax1.set_ylabel('R$')
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

# Painel 2: Custo/HL vs Margem Bruta
ax2 = fig.add_subplot(gs[0, 1])
custo_hl = df_cen['Custo_HL'].values
margem_hl = df_cen['Margem_HL'].values
for i, (n, c, m, cor) in enumerate(zip(nomes_cen, custo_hl, margem_hl, cores_cen)):
    ax2.bar(i - 0.2, c, 0.35, color=cor, alpha=0.85, label=f'Custo/HL {n}')
    ax2.bar(i + 0.2, m, 0.35, color=cor, alpha=0.4, hatch='//')
ax2.axhline(136, color=CORES['azul_escuro'], linestyle='--', linewidth=2, label='Margem bruta base (R$136/HL)')
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(['A — Realocação', 'B — Rodoviário', 'C — Combinação'])
ax2.set_ylabel('R$/HL')
ax2.set_title('Custo/HL (sólido) vs Margem Residual/HL (hachura)', fontsize=10)
ax2.legend(fontsize=7)

# Painel 3: Lead Time vs DOI Mínimo
ax3 = fig.add_subplot(gs[1, 0])
lead_times = df_cen['Lead_Time'].values
doi_patagonia_com_acao = [5.5, 28.0, 14.2]  # estimativas de DOI Patagonia pós-ação
for i, (n, lt, dp, cor) in enumerate(zip(nomes_cen, lead_times, doi_patagonia_com_acao, cores_cen)):
    ax3.scatter(lt, dp, s=300, color=cor, zorder=5, label=f'Cenário {n}')
    ax3.annotate(f'Cenário {n}\nLT={lt}d, DOI Pat.={dp:.1f}d',
                xy=(lt, dp), xytext=(lt+0.3, dp+1), fontsize=8, color=cor)
ax3.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI Mínimo ({DOI_MIN}d)')
ax3.fill_between([0, 30], [0, 0], [DOI_MIN, DOI_MIN], alpha=0.1, color=CORES['vermelho'])
ax3.set_xlabel('Lead Time (dias)')
ax3.set_ylabel('DOI Patagonia pós-ação (dias)')
ax3.set_title('Lead Time vs. DOI Patagonia por Cenário', fontsize=10)
ax3.legend(fontsize=8)
ax3.set_xlim(0, 30)

# Painel 4: Radar
ax4 = fig.add_subplot(gs[1, 1], polar=True)
categorias = ['Custo\n(inv)', 'Velocidade', 'Margem', 'Flexibilidade', 'Risco\n(inv)']
N = len(categorias)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

# Normalizar: custo (invertido), velocidade, margem (normalizada), flexibilidade, risco (invertido)
max_custo = 596760
cen_radar = {
    'A — Realocação': [
        (1 - 321336/max_custo) * 5,  # custo invertido
        1,   # velocidade baixa
        df_cen.loc['A — Realocação', 'Margem_HL'] / 285 * 5,
        1,   # flexibilidade baixa
        (5 - 3),  # risco invertido
    ],
    'B — Rodoviário': [
        (1 - 596760/max_custo) * 5,
        5, df_cen.loc['B — Rodoviário', 'Margem_HL'] / 285 * 5,
        3, (5 - 2),
    ],
    'C — Combinação': [
        (1 - 458858/max_custo) * 5,
        5, df_cen.loc['C — Combinação', 'Margem_HL'] / 285 * 5,
        5, (5 - 1),
    ],
}

for nome, cor in zip(cen_radar.keys(), cores_cen):
    vals = cen_radar[nome] + [cen_radar[nome][0]]
    ax4.plot(angles, vals, color=cor, linewidth=2.5, label=nome)
    ax4.fill(angles, vals, color=cor, alpha=0.15)

ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(categorias, fontsize=8)
ax4.set_ylim(0, 5)
ax4.set_title('Radar — Cenários em 5 Dimensões', fontsize=10, pad=15)
ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

plt.savefig('fig_bivariada_36.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico

**Painel superior esquerdo — Custo Total:**
- Barras coloridas = custo total de cada cenário (quanto Ambev gasta)
- Linha verde tracejada = receita total esperada (MACO × volume = R$1.282.500)
- Quanto menor a barra, maior a margem restante
- Cenário A é o mais barato, mas vem com risco de DOI e lead de 25 dias

**Painel superior direito — Custo/HL vs Margem Residual/HL:**
- Barras **sólidas** = custo de frete por HL para cada cenário
- Barras com **hachura** = margem residual que sobra após deduzir frete
- Linha tracejada = margem bruta base de R$136/HL (sem frete)
- Quanto maior a diferença entre sólida e hachura, melhor o cenário

**Painel inferior esquerdo — Lead Time vs DOI Patagonia:**
- Eixo X = tempo até o produto Malzbier chegar ao CDR (dias)
- Eixo Y = DOI resultante para Patagonia (afetado pela realocação)
- Faixa vermelha = zona de ruptura de Patagonia (DOI < 12d)
- Cenário A fica na zona vermelha — lead alto + DOI Patagonia baixo
- Cenários B e C ficam acima — rápidos e com Patagonia seguro

**Painel inferior direito — Radar 5 Dimensões:**
- Área **maior** = melhor desempenho geral (exceto Custo e Risco, que são invertidos)
- Cenário C (verde) tem a maior área — domina em Velocidade, Flexibilidade e Risco
- Cenário B (vermelho) perde em Custo — área grande mas concentrada nos eixos errados

> 💡 **Cenário C domina: 23% mais barato que B, mesmo lead time (6 dias), mesma segurança de DOI. Apenas reduz Patagonia em -2.250 HL (vs -4.500 no A), mantendo DOI Patagonia acima do mínimo.**


---
## 37. Recomendação Final — Cenário C: Combinação

> **✅ IMPLEMENTAR CENÁRIO C — COMBINAÇÃO: 50% Realocação PE541 + 50% Rodoviário SP→BA**
>
> **Custo: R$458.858 | Lead: 6 dias | DOI Risk: Baixo | ROI Mar-Jun: ~730%**

### 5 Razões para o Cenário C

1. **Velocidade:** Lead time de 6 dias garante produto em fevereiro — antes que Mapapi e NE Norte zerem o estoque
2. **Custo:** 23% mais barato que o Rodoviário puro (B), economizando R$137.902
3. **Margem:** R$182,99/HL de margem residual vs. R$152,39/HL no B
4. **Resiliência ao BIAS:** Mantém retorno positivo até BIAS de -18% (vs. -4,7% no B)
5. **Patagonia protegida:** Reduz Patagonia apenas em -2.250 HL (metade do A), DOI permanece acima de 12 dias

### O que acontece se NÃO agirmos?

| Sub-Região | Ação (Cenário C) | Inação |
|---|---|---|
| Mapapi (34,6%) | DOI recuperado para 15+ dias em W1 | **Ruptura imediata (DOI=-3,2d)** — perda de R$443k |
| NE Norte (15,4%) | DOI recuperado para 14+ dias em W1 | **Ruptura imediata (DOI=-2,3d)** — perda de R$197k |
| NO Centro (16,8%) | DOI estabilizado em 14+ dias | Ruptura em W2 — perda de R$215k |
| NE Sul (24,1%) | DOI mantido em 16+ dias | Alerta W3, ruptura W4 — perda de R$308k |

**Custo total da inação estimado: R$1.163.000 em receita perdida (4.500 HL × R$285/HL × % não atendido)**

### Plano de Ação Semanal

| Semana | Data | Ação | Responsável | Entregável |
|---|---|---|---|---|
| W0 | 02/02 (D+0) | Aprovar Cenário C em S&OP emergencial | Diretoria NENO | Ata de aprovação |
| W0 | 02/02 (D+0) | Emitir NF rodoviária SP→BA (2.363 HL bruto) | Logística/PCP | NF emitida |
| W0 | 03/02 (D+1) | Reprojetar PE541: +2.250 HL Malzbier, -2.250 HL Patagonia W1 | PCP PE541 | Plano PCP atualizado |
| W1 | 05/02 (D+3) | Receber realocação em CDR João Pessoa (2.250 HL) | Logística NENO | Recebimento confirmado |
| W1 | 08/02 (D+6) | Receber rodoviário em CDR Camaçari/João Pessoa (2.250 HL liq.) | Logística NENO | Recebimento confirmado |
| W2 | 09/02 | Verificar DOI por sub-região — todos devem estar >12d | Comercial/Financeiro | Relatório DOI |
| W2-W4 | 16-28/02 | Monitorar DOI semanal e ajustar se necessário | S&OP NENO | Dashboard semanal |
| W4+ | 28/02+ | Avaliar demanda real vs forecast — recalibrar BIAS | Finance/S&OP | Análise de BIAS |

### KPIs de Controle

| KPI | Meta | Alerta | Crítico |
|---|---|---|---|
| DOI Malzbier NENO | ≥ 15 dias | < 12 dias | < 7 dias |
| DOI Patagonia NENO | ≥ 12 dias | < 10 dias | < 7 dias |
| Custo frete realizado | ≤ R$458.858 | > R$500k | > R$600k |
| Volume Malzbier entregue | ≥ 4.500 HL | < 4.000 HL | < 3.500 HL |
| BIAS realizado fev. | < ±5% | 5%-10% | > 10% |

### Registro de Riscos

| Risco | Probabilidade | Impacto | Mitigação |
|---|---|---|---|
| Patagonia abaixo do DOI mínimo | Média | Alto | Monitorar diariamente; priorizar cabotagem compensação em março |
| Rodoviário com mais de 5% de quebra | Baixa | Médio | Contratar seguro de carga; inspecionar veículos |
| Demanda real 15% abaixo do forecast | Média | Médio | Revisar pedidos semanalmente; flexibilidade de cancelamento D+2 |
| Concorrente ocupa gôndola durante ruptura | Alta (sem ação) | Muito Alto | Garantir estoque mínimo em PDV críticos |
| Goose Island sem produção em PE541 | Baixa | Muito Alto | Confirmar restrição de elaboração; plano B: AQ541 |

### Posicionamento para Março e Além

A demanda de +10%/mês a partir de março significa que em abril (Mar × 1,1) a demanda já supera toda a capacidade NE disponível. A **próxima decisão estratégica** é:
1. Contratar capacidade adicional na PE541 (expansão de linha)
2. Negociar acordo de fornecimento inter-regional permanente com SP
3. Revisar o mix de produção PE541 para maximizar Malzbier nos meses de alta demanda


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Seção 37 — Dashboard Executivo Final — Cenário C
# ═══════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(20, 14))
gs_dash = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Seção 37 — Dashboard Executivo: Recomendação Final — Cenário C',
             fontsize=14, fontweight='bold', color=CORES['azul_escuro'])
fig.patch.set_facecolor(CORES['bg'])

# ── Painel 1 (topo esquerdo): KPI Box ────────────────────────────────────
ax1 = fig.add_subplot(gs_dash[0, 0])
ax1.set_facecolor(CORES['azul_escuro'])
ax1.axis('off')

kpis = [
    ('CENÁRIO RECOMENDADO', 'C — COMBINAÇÃO', CORES['ambar']),
    ('Volume Malzbier', '4.500 HL/mês', CORES['branco']),
    ('Custo Total', 'R$ 458.858', CORES['branco']),
    ('Custo/HL', 'R$ 102,01', CORES['branco']),
    ('Lead Time', '6 dias (D+6)', CORES['verde']),
    ('DOI Risk', 'BAIXO', CORES['verde']),
    ('Margem Residual/HL', 'R$ 182,99', CORES['verde']),
    ('ROI Mar-Jun', '~730%', CORES['ambar']),
    ('Payback', 'Março', CORES['ambar']),
]

for i, (label, val, color) in enumerate(kpis):
    y = 0.95 - i * 0.105
    ax1.text(0.03, y, label + ':', transform=ax1.transAxes,
             fontsize=8.5, color=CORES['cinza_medio'], va='top')
    ax1.text(0.97, y, val, transform=ax1.transAxes,
             fontsize=9, color=color, va='top', ha='right', fontweight='bold')
ax1.set_title('KPIs — Cenário C', fontsize=10, color=CORES['branco'],
              pad=5, fontweight='bold')

# ── Painel 2 (topo centro): Comparação cenários ───────────────────────────
ax2 = fig.add_subplot(gs_dash[0, 1])
cen_names = ['A', 'B', 'C']
custos_all = [321336, 596760, 458858]
margens_all = [(285 - 71.41) * 4500, (285 - 132.61) * 4500, (285 - 102.01) * 4500]

x_c = np.arange(3)
bars_c = ax2.bar(x_c, custos_all, 0.5, color=cores_cen, alpha=0.85, label='Custo Total')
bars_m = ax2.bar(x_c, margens_all, 0.5, bottom=custos_all, color=cores_cen, alpha=0.35, hatch='//', label='Margem Residual')
for i, (c, m) in enumerate(zip(custos_all, margens_all)):
    ax2.text(i, c/2, f'R${c/1000:.0f}k', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    ax2.text(i, c + m/2, f'R${m/1000:.0f}k', ha='center', va='center', fontsize=7, color=cores_cen[i])
ax2.set_xticks(x_c)
ax2.set_xticklabels(['A — Realoc.', 'B — Rodo', 'C — Comb.'])
ax2.set_ylabel('R$')
ax2.set_title('Custo (sólido) + Margem (hachura)', fontsize=10)
ax2.legend(fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

# ── Painel 3 (topo direito): Mapa geográfico por tratamento ──────────────
ax3 = fig.add_subplot(gs_dash[0, 2])
ax3.axis('off')
geo_info = [
    ['Sub-Região', '% Dem', 'DOI W1', 'Ação Cenário C', 'Prioridade'],
    ['Mapapi', '34,6%', '-3,2d', 'Realoc. + Rodo → JP', '🔴 1'],
    ['NE Norte', '15,4%', '-2,3d', 'Realoc. + Rodo → JP', '🔴 2'],
    ['NO Centro', '16,8%', '4,8d', 'Rodo → JP', '🟠 3'],
    ['NE Sul', '24,1%', '9,8d', 'Rodo → Camaçari', '🟡 4'],
    ['NO Araguaia', '0,6%', '-6,3d', 'DESPRIORITIZAR', '⚪ 5'],
]
tbl3 = ax3.table(cellText=geo_info[1:], colLabels=geo_info[0], loc='center', cellLoc='center')
tbl3.auto_set_font_size(False)
tbl3.set_fontsize(8.5)
tbl3.scale(1.1, 1.5)
row_colors3 = ['#FFCCCC', '#FFCCCC', '#FFE0B2', '#FFF9C4', '#F5F5F5']
for i, rc in enumerate(row_colors3):
    for j in range(5):
        tbl3[i+1, j].set_facecolor(rc)
for j in range(5):
    tbl3[0, j].set_facecolor(CORES['azul_escuro'])
    tbl3[0, j].set_text_props(color='white', fontweight='bold')
ax3.set_title('Ações por Sub-Região — Cenário C', fontsize=10, pad=20)

# ── Painel 4 (baixo esquerdo): DOI evolução ───────────────────────────────
ax4 = fig.add_subplot(gs_dash[1, 0])
sub_regioes_plot = ['Mapapi', 'NE Norte', 'NO Centro', 'NE Sul']
for i, sr in enumerate(sub_regioes_plot):
    ax4.plot(semana_nums, doi_baseline[sr], 'o--', color=cores_sr[i], linewidth=1.5, alpha=0.5)
    ax4.plot(semana_nums, doi_cenC[sr], 's-', color=cores_sr[i], linewidth=2.5, label=sr)
ax4.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI Min ({DOI_MIN}d)')
ax4.fill_between([0.8, 4.2], [0, 0], [DOI_MIN, DOI_MIN], alpha=0.07, color=CORES['vermelho'])
ax4.set_xticks(semana_nums)
ax4.set_xticklabels(w_labels)
ax4.set_ylabel('DOI (dias)')
ax4.set_title('DOI — Sem Ação (tracejado) vs Cenário C (sólido)', fontsize=9)
ax4.legend(fontsize=7, ncol=2)

# ── Painel 5 (baixo centro): Gantt de ações ───────────────────────────────
ax5 = fig.add_subplot(gs_dash[1, 1])
acoes = [
    ('Aprovação S&OP', 0, 1, CORES['azul_escuro']),
    ('Emitir NF Rodoviário', 0, 1, CORES['ambar']),
    ('Reprojetar PE541', 1, 2, CORES['azul_medio']),
    ('Produção Malzbier PE541', 2, 5, CORES['azul_claro']),
    ('Transporte Rodoviário SP→BA', 1, 6, CORES['ambar']),
    ('Recebimento JP (Realoc.)', 3, 4, CORES['verde']),
    ('Recebimento CDRs (Rodo)', 6, 7, CORES['verde']),
    ('Distribuição PDV', 7, 14, CORES['verde']),
    ('Monitoramento DOI W2-W4', 7, 28, CORES['cinza_medio']),
    ('Revisão BIAS pós-fev.', 28, 32, CORES['azul_escuro']),
]
for i, (nome, inicio, fim, cor) in enumerate(acoes):
    ax5.barh(i, fim - inicio, left=inicio, height=0.5, color=cor, alpha=0.85)
    ax5.text(inicio + (fim - inicio) / 2, i, nome, ha='center', va='center',
             fontsize=6.5, color='white', fontweight='bold')
ax5.axvline(0, color=CORES['azul_escuro'], linewidth=1.5, linestyle='-', label='D+0 (02/02)')
ax5.axvline(6, color=CORES['verde'], linewidth=1.5, linestyle='--', label='D+6: Rodo chega')
ax5.axvline(28, color=CORES['ambar'], linewidth=1.5, linestyle=':', label='D+28: Fim fev.')
ax5.set_xlabel('Dias após D+0 (02/02/2026)')
ax5.set_yticks([])
ax5.set_title('Timeline de Ações — D+0 a D+32', fontsize=9)
ax5.legend(fontsize=7)

# ── Painel 6 (baixo direito): Árvore de decisão textual ──────────────────
ax6 = fig.add_subplot(gs_dash[1, 2])
ax6.set_facecolor('#F8F9FA')
ax6.axis('off')
decision_text = (
    "ÁRVORE DE DECISÃO\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    "Gap = 4.500 HL/mês (confirmado)\n"
    "        │\n"
    "        ├─ Lead time aceitável (<7d)?\n"
    "        │        └─ SIM → Rodo ou Combinar\n"
    "        │\n"
    "        ├─ Margem > R$100/HL necessária?\n"
    "        │        └─ SIM → NÃO Rodo puro (B)\n"
    "        │\n"
    "        ├─ DOI Patagonia seguro?\n"
    "        │        └─ SIM → NÃO Realoc. pura (A)\n"
    "        │\n"
    "        └─ RESULTADO: CENÁRIO C\n"
    "             50% Realoc. + 50% Rodo\n"
    "             Custo: R$458.858\n"
    "             Lead: 6 dias\n"
    "             DOI Risk: BAIXO\n"
    "             ✅ IMPLEMENTAR"
)
ax6.text(0.05, 0.95, decision_text, transform=ax6.transAxes,
         fontsize=8.5, va='top', ha='left', family='monospace',
         color=CORES['azul_escuro'],
         bbox=dict(boxstyle='round,pad=0.5', facecolor=CORES['cinza_claro'], alpha=0.8))
ax6.set_title('Árvore de Decisão — Lógica do Cenário C', fontsize=9)

plt.savefig('fig_bivariada_37.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Como Ler Este Gráfico (Dashboard Executivo)

**Painel KPI (topo esquerdo):**
- Fundo escuro com texto claro = painel executivo de referência rápida
- Valores em **verde** = métricas favoráveis (DOI Baixo, Lead 6 dias, ROI 730%)
- Valores em **âmbar** = métricas de atenção (custo, retorno)
- Os 6 dias de lead time são críticos: qualquer opção com >7 dias não salva fevereiro

**Comparação de cenários (topo centro):**
- Barras **sólidas** = custo (o que você paga)
- Barras com **hachura** (sobre as sólidas) = margem residual (o que você retém)
- Cenário C tem barra de custo moderada E margem residual maior que B

**Mapa geográfico (topo direito):**
- **Vermelho** = ruptura ativa (ação D+0 obrigatória)
- **Laranja** = crítico (ação D+3)
- **Amarelo** = alerta (monitorar)
- As ações na coluna "Cenário C" mostram como cada sub-região é atendida: realocação e rodoviário chegam por rotas diferentes (Fonte Mata/João Pessoa vs. Camaçari)

**DOI Evolution (baixo esquerdo):**
- Linhas **tracejadas** = destino sem ação (ruptura confirmada)
- Linhas **sólidas** = com Cenário C implementado
- Todas as linhas sólidas ficam acima de 12 dias — meta atingida
- Semana W2 é o ponto crítico: sem ação, Mapapi e NE Norte já estão em zero

**Gantt de ações (baixo centro):**
- Mostra o caminho crítico: aprovação D+0 → NF D+0 → reprojeção D+1 → chegadas D+3/D+6
- As ações paralelas (realocação e rodoviário simultaneamente) permitem o lead de 6 dias
- Monitoramento DOI se estende por todo fevereiro como ação contínua

**Árvore de decisão (baixo direito):**
- Lógica de eliminação: A e B são eliminados por critérios específicos
- Cenário C não é "o menos ruim" — é genuinamente superior em múltiplos critérios
- A árvore mostra que qualquer outra escolha falha em pelo menos uma dimensão crítica

---

## Próximos Passos Imediatos (48h)

1. **D+0 (02/02):** Convocar S&OP emergencial NENO — apresentar este dashboard — aprovar Cenário C
2. **D+0 (02/02):** PCP PE541 reproduzir plano revisado com +2.250 HL Malzbier W1
3. **D+0 (02/02):** Logística emitir NF para 2.363 HL Malzbier via rodoviário SP→BA

## Próximos Passos Estratégicos (30-60 dias)

1. **Fev → Mar:** Monitorar BIAS realizado; se demanda confirmar +30%, escalar produção PE541 para W3-W4
2. **Mar:** Avaliar cabotagem recorrente para compensar capacidade de Patagonia deslocada
3. **Abr:** Iniciar projeto de expansão de capacidade Malzbier na NENO — demanda ultrapassa cap. disponível em abril segundo projeção de +10%/mês
